# Code as a Judge — Custom Evaluation with Plain Functions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fasteval/fasteval/blob/main/docs/notebooks/code-as-judge.ipynb)

This notebook walks through **Code as a Judge** — fasteval's way of turning a plain Python function into a reusable evaluation metric. No base classes, no boilerplate.

**What you'll learn:**

1. Writing scoring functions that return a float between 0 and 1
2. Using `@fe.judge()` to wire them into the evaluation pipeline
3. Different return types (float, tuple, dict) for richer feedback
4. Flexible function signatures — pick the fields you need
5. Combining code judges with built-in metrics
6. Real-world examples: format validation, PII detection, response quality

**No API key required.** Every cell in this notebook is runnable as-is.

---

## Setup

In [ ]:
!pip install -q fasteval-core

In [ ]:
import fasteval as fe
from fasteval import EvalInput, MetricResult

print(f"fasteval {fe.__version__} ready")

---

## 1. Your First Code Judge

A code judge is a plain function that:
- Takes evaluation data as arguments (e.g., `actual_output`, `expected_output`)
- Returns a score between **0.0** and **1.0**

You pass it to `@fe.judge()` and fasteval does the rest.

In [ ]:
# Step 1: Define a plain scoring function
def check_word_count(actual_output: str) -> float:
    """Penalize responses that are too short."""
    words = len((actual_output or "").split())
    if words < 5:
        return 0.2
    if words > 200:
        return 0.4
    return 1.0


# Step 2: Use it with @fe.judge on a test function
@fe.judge(check_word_count, threshold=0.5)
def test_response_length():
    response = "The capital of France is Paris, a beautiful city in Western Europe."
    return fe.score(response)


result = test_response_length()
print(f"Passed: {result.passed}")
print(f"Score:  {result.metric_results[0].score}")
print(f"Metric: {result.metric_results[0].metric_name}")

That's it. The function name (`check_word_count`) automatically becomes the metric name. No subclassing, no registration.

---

## 2. Return Types — Adding Reasoning and Details

A bare `float` works, but you can return richer information that shows up in failure reports.

### 2a. Tuple — `(score, reasoning)`

In [ ]:
def check_politeness(actual_output: str) -> tuple:
    """Check if the response uses polite language."""
    polite_words = ["please", "thank you", "sorry", "appreciate", "kind"]
    text = (actual_output or "").lower()
    found = [w for w in polite_words if w in text]

    if len(found) >= 2:
        return (1.0, f"Polite language found: {found}")
    if len(found) == 1:
        return (0.6, f"Somewhat polite — found: {found}")
    return (0.1, "No polite language detected")


@fe.judge(check_politeness, threshold=0.5)
def test_polite_response():
    response = "Thank you for reaching out! I appreciate your patience."
    return fe.score(response)


result = test_polite_response()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")

### 2b. Dict — `{"score", "reasoning", "details"}`

In [ ]:
import json


def check_json_structure(actual_output: str) -> dict:
    """Validate that the response is JSON with required fields."""
    required_fields = {"id", "status", "message"}
    try:
        data = json.loads(actual_output)
    except (json.JSONDecodeError, TypeError):
        return {"score": 0.0, "reasoning": "Invalid JSON", "details": {"parse_error": True}}

    present = required_fields & set(data.keys())
    missing = required_fields - present

    score = len(present) / len(required_fields)
    return {
        "score": score,
        "reasoning": f"Found {len(present)}/{len(required_fields)} required fields"
                     + (f" — missing: {missing}" if missing else ""),
        "details": {"present": list(present), "missing": list(missing)},
    }


@fe.judge(check_json_structure, threshold=1.0)
def test_api_response_format():
    response = json.dumps({"id": 42, "status": "ok", "message": "Order placed"})
    return fe.score(response)


result = test_api_response_format()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")
print(f"Details:   {mr.details}")

### 2c. MetricResult — Full Control

In [ ]:
def check_sentiment(actual_output: str) -> MetricResult:
    """Simple keyword-based sentiment check with full MetricResult."""
    text = (actual_output or "").lower()
    positive = ["great", "excellent", "happy", "good", "wonderful"]
    negative = ["bad", "terrible", "awful", "horrible", "worst"]

    pos_count = sum(1 for w in positive if w in text)
    neg_count = sum(1 for w in negative if w in text)
    total = pos_count + neg_count
    score = pos_count / total if total > 0 else 0.5

    return MetricResult(
        metric_name="sentiment",
        score=score,
        passed=score >= 0.6,
        threshold=0.6,
        reasoning=f"{pos_count} positive, {neg_count} negative keywords",
        details={"positive_count": pos_count, "negative_count": neg_count},
    )


@fe.judge(check_sentiment, threshold=0.6)
def test_positive_tone():
    response = "That's a great question! I'm happy to help with this wonderful topic."
    return fe.score(response)


result = test_positive_tone()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")
print(f"Details:   {mr.details}")

---

## 3. Function Signature Flexibility

Your scoring function declares what data it needs via parameter names. fasteval matches them against `EvalInput` fields.

### 3a. Multiple Fields

In [ ]:
def check_answer_contains_expected(
    actual_output: str, expected_output: str, input: str
) -> tuple:
    """Check if the expected answer appears in the actual output."""
    actual = (actual_output or "").lower()
    expected = (expected_output or "").lower()

    if expected in actual:
        return (1.0, f"Expected answer '{expected_output}' found in response")
    return (0.0, f"Expected '{expected_output}' not found in response to '{input}'")


@fe.judge(check_answer_contains_expected, threshold=1.0)
def test_qa_answer():
    return fe.score(
        "The capital of France is Paris.",
        "Paris",
        input="What is the capital of France?",
    )


result = test_qa_answer()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")

### 3b. Full EvalInput Access

If you need everything (context, tool calls, metadata, etc.), declare an `eval_input` parameter.

In [ ]:
def comprehensive_check(eval_input: EvalInput) -> tuple:
    """Run multiple checks using the full evaluation input."""
    issues = []

    if not eval_input.actual_output:
        issues.append("empty response")
    elif len(eval_input.actual_output.split()) < 3:
        issues.append("response too short")

    if eval_input.expected_output and eval_input.actual_output:
        if eval_input.expected_output.lower() not in eval_input.actual_output.lower():
            issues.append("expected content missing")

    if eval_input.context:
        ctx_text = " ".join(eval_input.context).lower()
        response_words = (eval_input.actual_output or "").lower().split()
        overlap = sum(1 for w in response_words if w in ctx_text)
        if overlap / max(len(response_words), 1) < 0.1:
            issues.append("response not grounded in context")

    score = max(1.0 - (len(issues) * 0.3), 0.0)
    reasoning = "All checks passed" if not issues else f"Issues: {', '.join(issues)}"
    return (score, reasoning)


@fe.judge(comprehensive_check, threshold=0.7)
def test_full_eval():
    return fe.score(
        "Paris is the capital of France, located in Western Europe.",
        "Paris",
        input="What is the capital of France?",
        context=["France is a country in Western Europe. Its capital is Paris."],
    )


result = test_full_eval()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")

### 3c. Using **kwargs

Name the fields you care about, and let `**kwargs` catch the rest.

In [ ]:
def check_not_empty(actual_output: str, **kwargs) -> float:
    """Simply check the response isn't empty."""
    return 1.0 if actual_output and actual_output.strip() else 0.0


@fe.judge(check_not_empty, threshold=1.0)
def test_non_empty():
    return fe.score("Hello, world!")


result = test_non_empty()
print(f"Passed: {result.passed}, Score: {result.metric_results[0].score}")

---

## 4. Combining Code Judges with Built-in Metrics

Stack `@fe.judge()` with any built-in metric. Code judges run alongside deterministic and LLM-based metrics.

In [ ]:
def no_profanity(actual_output: str) -> tuple:
    """Check that response doesn't contain profanity."""
    bad_words = ["damn", "hell", "crap"]
    text = (actual_output or "").lower()
    found = [w for w in bad_words if w in text]
    if found:
        return (0.0, f"Profanity detected: {found}")
    return (1.0, "Clean language")


def reasonable_length(actual_output: str) -> tuple:
    """Ensure response is between 10 and 300 words."""
    words = len((actual_output or "").split())
    if words < 10:
        return (0.2, f"Too short: {words} words")
    if words > 300:
        return (0.3, f"Too long: {words} words")
    return (1.0, f"Good length: {words} words")


# Stack multiple code judges with a built-in deterministic metric
@fe.judge(no_profanity, threshold=1.0)
@fe.judge(reasonable_length, threshold=0.5)
@fe.contains()
def test_support_response():
    response = (
        "Thank you for contacting our support team. "
        "I understand your concern about the billing issue. "
        "We have reviewed your account and found that the charge "
        "was processed correctly. However, I've applied a courtesy "
        "credit to your account. Please allow 3-5 business days "
        "for it to appear on your statement."
    )
    return fe.score(response, "billing")


result = test_support_response()
print(f"Overall passed: {result.passed}")
print(f"Aggregate score: {result.aggregate_score:.2f}")
print(f"---")
for mr in result.metric_results:
    print(f"  {mr.metric_name}: {mr.score:.2f} {'PASS' if mr.passed else 'FAIL'}")
    if mr.reasoning:
        print(f"    -> {mr.reasoning}")

---

## 5. Real-World Examples

### 5a. PII Detection

In [ ]:
import re


def no_pii(actual_output: str) -> dict:
    """Detect common PII patterns in the response."""
    text = actual_output or ""
    findings = {}

    if re.search(r"\b\d{3}-\d{2}-\d{4}\b", text):
        findings["ssn"] = True
    if re.search(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", text):
        findings["email"] = True
    if re.search(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b", text):
        findings["phone"] = True
    if re.search(r"\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b", text):
        findings["credit_card"] = True

    if findings:
        return {
            "score": 0.0,
            "reasoning": f"PII detected: {list(findings.keys())}",
            "details": findings,
        }
    return {"score": 1.0, "reasoning": "No PII detected", "details": {}}


# Test with a clean response
@fe.judge(no_pii, threshold=1.0)
def test_no_pii_leak():
    response = "Your account has been updated. Please check your dashboard for details."
    return fe.score(response)


result = test_no_pii_leak()
print(f"Passed: {result.passed}")
print(f"Reasoning: {result.metric_results[0].reasoning}")

In [ ]:
# Now test with a response that leaks PII — this should FAIL
@fe.judge(no_pii, threshold=1.0)
def test_pii_leak():
    response = "Your SSN is 123-45-6789 and your email is john@example.com."
    return fe.score(response)


try:
    test_pii_leak()
except fe.EvaluationFailedError as e:
    mr = e.result.metric_results[0]
    print(f"Test correctly failed!")
    print(f"Score:   {mr.score}")
    print(f"Reason:  {mr.reasoning}")
    print(f"Details: {mr.details}")

### 5b. Response Format Compliance

In [ ]:
def has_required_sections(actual_output: str) -> dict:
    """Check that a support response contains required sections."""
    text = (actual_output or "").lower()
    sections = {
        "greeting": any(g in text for g in ["hello", "hi", "thank you for", "thanks for"]),
        "action_taken": any(a in text for a in ["i have", "i've", "we have", "we've", "we will"]),
        "next_steps": any(n in text for n in ["please", "next step", "you can", "follow up"]),
    }

    present = sum(sections.values())
    total = len(sections)
    missing = [k for k, v in sections.items() if not v]

    return {
        "score": present / total,
        "reasoning": f"{present}/{total} sections present"
                     + (f" — missing: {missing}" if missing else ""),
        "details": sections,
    }


@fe.judge(has_required_sections, threshold=1.0)
def test_support_format():
    response = (
        "Hello! Thank you for reaching out.\n\n"
        "I've looked into your issue and found the root cause.\n\n"
        "Please restart the application and follow up if the issue persists."
    )
    return fe.score(response)


result = test_support_format()
mr = result.metric_results[0]
print(f"Score:     {mr.score}")
print(f"Reasoning: {mr.reasoning}")
print(f"Details:   {mr.details}")

### 5c. RAG Grounding Check

In [ ]:
def grounding_overlap(actual_output: str, context: list) -> tuple:
    """Measure how much of the response overlaps with the provided context."""
    if not context:
        return (0.5, "No context provided")

    ctx_words = set()
    for doc in context:
        ctx_words.update(doc.lower().split())

    # Filter out common stop words for a more meaningful check
    stop_words = {"the", "a", "an", "is", "in", "of", "and", "to", "for", "it", "on", "that", "this", "with"}
    response_words = [w.strip(".,!?") for w in (actual_output or "").lower().split()]
    meaningful_words = [w for w in response_words if w and w not in stop_words]

    if not meaningful_words:
        return (0.0, "Response has no meaningful words")

    grounded = sum(1 for w in meaningful_words if w in ctx_words)
    ratio = grounded / len(meaningful_words)

    return (min(ratio * 1.5, 1.0), f"{grounded}/{len(meaningful_words)} content words grounded in context")


@fe.judge(grounding_overlap, threshold=0.5)
def test_rag_grounding():
    context = [
        "Python was created by Guido van Rossum in 1991.",
        "Python emphasizes code readability and simplicity.",
        "Python supports multiple programming paradigms.",
    ]
    response = (
        "Python was created by Guido van Rossum. "
        "It emphasizes code readability and supports multiple programming paradigms."
    )
    return fe.score(response, context=context)


result = test_rag_grounding()
mr = result.metric_results[0]
print(f"Score:     {mr.score:.2f}")
print(f"Reasoning: {mr.reasoning}")

---

## 6. Testing Your Scoring Functions Independently

Since scoring functions are plain functions, you can test them directly — no fasteval setup required.

In [ ]:
# Unit test your scoring function directly
assert check_word_count("short") == 0.2
assert check_word_count("This is a sentence with enough words") == 1.0
assert check_word_count(" ".join(["word"] * 250)) == 0.4

# Test tuple returns
score, reasoning = check_politeness("Thank you so much, I really appreciate it!")
assert score == 1.0
assert "thank you" in reasoning.lower() or "appreciate" in reasoning.lower()

score, reasoning = check_politeness("Whatever.")
assert score == 0.1

# Test dict returns
result = no_pii("Call me at 555-123-4567")
assert result["score"] == 0.0
assert result["details"]["phone"] is True

result = no_pii("Everything looks good on your account.")
assert result["score"] == 1.0

print("All unit tests passed!")

---

## 7. Handling Failures Gracefully

When a code judge fails its threshold, fasteval raises `EvaluationFailedError`. The error contains the full result so you can inspect what went wrong.

In [ ]:
def strict_length(actual_output: str) -> tuple:
    words = len((actual_output or "").split())
    if words < 20:
        return (0.0, f"Only {words} words — need at least 20")
    return (1.0, f"{words} words — meets minimum")


@fe.judge(strict_length, threshold=1.0)
def test_strict():
    return fe.score("Too short.")


try:
    test_strict()
except fe.EvaluationFailedError as e:
    print("Evaluation failed (as expected):")
    print(f"  Aggregate score: {e.result.aggregate_score}")
    for mr in e.result.metric_results:
        print(f"  {mr.metric_name}: score={mr.score}, reasoning={mr.reasoning}")

---

## Summary

| Feature | How |
|---|---|
| Define a judge | Write a plain function returning `float`, `tuple`, `dict`, or `MetricResult` |
| Wire it in | `@fe.judge(fn, threshold=0.8)` on your test function |
| Pick your fields | Name parameters after `EvalInput` fields, or use `eval_input` for everything |
| Combine | Stack multiple `@fe.judge()` and built-in decorators freely |
| Test independently | Call your scoring function directly in unit tests |

For the full API reference, see the [Code as a Judge concept page](/docs/concepts/code-as-judge).